In [1]:
import numpy as np

In [2]:
class SkipgramModel:
    def __init__(self, vocab_size, embed_size, learning_rate = 0.1):
        self.v_size = vocab_size
        self.e_size = embed_size
        self.lr = learning_rate

        self.W1 = np.random.randn(self.v_size, self.e_size) * 0.1
        self.W2 = np.random.randn(self.e_size, self.v_size) * 0.1

    def _softmax(self, x):
        e_x = np.exp(x - np.max(x))
        
        return e_x / e_x.sum(axis = 0)

    def train_step(self, center_word_idx, context_word_indices):
        h = self.W1[center_word_idx]

        u = np.dot(h, self.W2)
        y_pred = self._softmax(u)

        total_error = np.zeros(self.v_size)
        for target_idx in context_word_indices:
            e = y_pred.copy()
            e[target_idx] -= 1
            total_error += e

        dW2 = np.outer(h, total_error)
        dW1 = np.dot(self.W2, total_error)

        self.W2 -= self.lr * dW2
        self.W1[center_word_idx] -= self.lr * dW1

        loss = -np.mean([np.log(y_pred[idx] + 1e-9) for idx in context_word_indices])

        return loss